# 02j — Training Pipeline (InceptionV3) with Synthetic Data

**Project:** UREP 32-0210-250078 | Crack Classification

**Classes:** Debonding | Flexural | Shear | Multi-Crack | No Crack | Others

**Experiment:** Same architecture, hyperparameters, and seeds as `02_training.ipynb`.
The only difference is that 806 synthetic AutoCAD-generated crack images are
added to the training set (102 debonding/corrosion, 302 flexural, 402 shear).
Hair Crack and Wide Crack variants are merged per type.
Val/test sets are identical to the baseline for fair comparison.

3-stage transfer learning with InceptionV3 (PyTorch):
1. **Stage 1** — Feature extraction (frozen backbone, LR=1e-3)
2. **Stage 2** — Partial fine-tuning (unfreeze from Mixed_7a, LR=1e-4)
3. **Stage 3** — Full fine-tuning (all layers, LR=1e-5)

**Metrics tracked per epoch:** loss, accuracy, precision, recall (train + val)

**Test evaluation:** Accuracy, Precision, Recall, F1-Score, IoU (per-class + mean)

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")

import os
import torch
import torch.nn as nn

import config
from src.dataset import prepare_dataset, split_dataset, get_dataloaders, compute_class_weights, prepare_synthetic_split
from src.model import InceptionV3Classifier, freeze_backbone, unfreeze_from, unfreeze_all
from src.trainer import train_model
from src.evaluation import plot_training_history, evaluate_model
from src.device import print_device_summary, get_device, set_seed

# Reproducibility
set_seed(config.RANDOM_SEED)

# Output directory for this model
OUTPUT_DIR = os.path.join(config.OUTPUT_DIR, "inceptionv3_synthetic")
os.makedirs(os.path.join(OUTPUT_DIR, "models"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "plots"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "logs"), exist_ok=True)

# Detect hardware
device_config = print_device_summary()
device = get_device()

STAGE_BATCH = device_config["batch_sizes"]
NUM_WORKERS = device_config["num_workers"]

print(f"\nDevice: {device}")
print(f"NUM_CLASSES: {config.NUM_CLASSES}")
print(f"CLASS_NAMES: {config.CLASS_NAMES}")
print(f"MAX_AUG_FACTOR: x{config.MAX_AUG_FACTOR}")

## Data Preparation (with Synthetic)

Materializes `split_synthetic/` directory: original train + 806 synthetic images.
Val/test are unchanged from the baseline split.

In [ ]:
# Ensure base split exists
if not os.path.exists(config.SPLIT_DIR) or not os.path.isdir(os.path.join(config.SPLIT_DIR, "train")):
    print("Preparing base dataset from QU structure...")
    prepare_dataset()
    split_dataset()

# Materialize synthetic split (original train + synthetic images)
SPLIT = config.SPLIT_SYNTHETIC_DIR
prepare_synthetic_split()
print(f"\nUsing split: {SPLIT}")

In [ ]:
# Create DataLoaders for Stage 1 (from synthetic split)
train_loader, val_loader, test_loader = get_dataloaders(
    split_dir=SPLIT,
    batch_size=STAGE_BATCH[1],
    img_size=config.IMG_SIZE,
    normalize="imagenet",
    num_workers=NUM_WORKERS,
)

In [ ]:
# Compute class weights from synthetic split training set
class_weight_dict = compute_class_weights(split_dir=SPLIT)
class_weight_tensor = torch.tensor(
    [class_weight_dict[i] for i in range(config.NUM_CLASSES)],
    dtype=torch.float32,
).to(device)
print(f"\nClass weight tensor: {class_weight_tensor}")

## Build Model

In [ ]:
model = InceptionV3Classifier().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
print(model)

## Stage 1 — Feature Extraction

Frozen InceptionV3 backbone, train only the classification head.

In [ ]:
freeze_backbone(model)
criterion = nn.CrossEntropyLoss(weight=class_weight_tensor)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=config.STAGE1_LR,
)

history1 = train_model(
    model, train_loader, val_loader, optimizer, criterion, device,
    epochs=config.STAGE1_EPOCHS,
    output_dir=OUTPUT_DIR,
    stage=1,
    model_name="inceptionv3_synth",
)

## Stage 2 — Partial Fine-Tuning

Unfreeze from `Mixed_7a` onwards, lower learning rate.

In [ ]:
# Recreate DataLoaders if batch size differs
if STAGE_BATCH[2] != STAGE_BATCH[1]:
    train_loader, val_loader, _ = get_dataloaders(
        split_dir=SPLIT,
        batch_size=STAGE_BATCH[2], img_size=config.IMG_SIZE,
        normalize="imagenet", num_workers=NUM_WORKERS,
    )

unfreeze_from(model, "Mixed_7a")
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=config.STAGE2_LR,
)

history2 = train_model(
    model, train_loader, val_loader, optimizer, criterion, device,
    epochs=config.STAGE2_EPOCHS,
    output_dir=OUTPUT_DIR,
    stage=2,
    model_name="inceptionv3_synth",
)

## Stage 3 — Full Fine-Tuning

All layers trainable, very low learning rate.

In [ ]:
if STAGE_BATCH[3] != STAGE_BATCH[2]:
    train_loader, val_loader, _ = get_dataloaders(
        split_dir=SPLIT,
        batch_size=STAGE_BATCH[3], img_size=config.IMG_SIZE,
        normalize="imagenet", num_workers=NUM_WORKERS,
    )

unfreeze_all(model)
optimizer = torch.optim.AdamW(model.parameters(), lr=config.STAGE3_LR)

history3 = train_model(
    model, train_loader, val_loader, optimizer, criterion, device,
    epochs=config.STAGE3_EPOCHS,
    output_dir=OUTPUT_DIR,
    stage=3,
    model_name="inceptionv3_synth",
)

## Training Curves

In [ ]:
plot_training_history(
    [history1, history2, history3],
    output_dir=OUTPUT_DIR,
    stage_names=["Feature Extraction", "Partial Fine-Tuning", "Full Fine-Tuning"],
    model_name="inceptionv3_synth",
)

## Save Final Model

In [ ]:
final_model_path = os.path.join(OUTPUT_DIR, "models", "best_model.pt")
torch.save(model.state_dict(), final_model_path)
print(f"Model saved to: {final_model_path}")

## Test Set Evaluation

Final evaluation on held-out test set (from **original** split, no synthetic images).
Computes: Accuracy, Precision, Recall, F1-Score, IoU.

In [ ]:
# Test from ORIGINAL split (no synthetic) for fair comparison
_, _, test_loader = get_dataloaders(
    split_dir=config.SPLIT_DIR,
    batch_size=STAGE_BATCH[1], img_size=config.IMG_SIZE,
    normalize="imagenet", num_workers=NUM_WORKERS,
)

metrics = evaluate_model(
    model, test_loader, device,
    output_dir=OUTPUT_DIR,
    model_name="inceptionv3_synth",
)